In [ ]:
%cd ..

In [ ]:
import glob
import json
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import HTML, display

In [ ]:
# Load all export files sorted by date
export_files = sorted(glob.glob("data/data/dataset_labeled*.json"))

# Load original dataset (from poe fetch_data)
import subprocess
import tempfile
import urllib.request

# Download original 136 labels to compare
url = "https://roai-data-readonly.s3.eu-central-1.amazonaws.com/dataset.json"
with urllib.request.urlopen(url) as r:
    original_dataset = json.loads(r.read())
print(f"Original dataset: {len(original_dataset)} labels")

# Build batches: original + each export's new keys
seen_keys = set(original_dataset.keys())
batches = [("Original (136 labels)", dict(original_dataset))]

for f in export_files:
    with open(f) as fh:
        export = json.load(fh)
    new_keys = {k: v for k, v in export.items() if k not in seen_keys}
    updated_keys = {k: v for k, v in export.items() if k in seen_keys}
    name = Path(f).stem.replace("dataset_labeled", "Batch")
    if not name.replace("Batch", "").strip("_"):
        name = "Batch 1 (first labeling)"
    else:
        name = f"Batch {name.replace('Batch_', '')}"
    batches.append((name, new_keys))
    seen_keys.update(export.keys())
    print(f"{Path(f).name}: {len(export)} total, {len(new_keys)} new, {len(updated_keys)} updated")

# Also check current dataset.json for any keys not in any export
with open("data/data/dataset.json") as fh:
    full_dataset = json.load(fh)
remaining = {k: v for k, v in full_dataset.items() if k not in seen_keys}
if remaining:
    batches.append(("Unlisted (in dataset.json only)", remaining))
    print(f"Unlisted keys in dataset.json: {len(remaining)}")

print(f"\nTotal batches: {len(batches)}")
for name, data in batches:
    print(f"  {name}: {len(data)} images")

In [ ]:
# Review images batch by batch
IMAGE_DIR = Path("data/data/minifigures")


def show_batch(name, data, cols=5):
    """Display a batch of labeled images in a grid."""
    keys = list(data.keys())
    if not keys:
        print(f"\n--- {name}: No images ---")
        return

    rows = (len(keys) + cols - 1) // cols
    fig, axs = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.5))
    if rows == 1:
        axs = [axs] if cols == 1 else [axs]
    axs = [ax for row in axs for ax in (row if hasattr(row, "__len__") else [row])]

    fig.suptitle(f"{name} ({len(keys)} images)", fontsize=14, fontweight="bold")

    for i, key in enumerate(keys):
        img_path = IMAGE_DIR / f"{key}.png"
        if img_path.exists():
            axs[i].imshow(plt.imread(img_path))
        labels_str = ", ".join(data[key])
        axs[i].set_title(f"{key}\n{labels_str}", fontsize=8)
        axs[i].axis("off")

    # Hide empty subplots
    for i in range(len(keys), len(axs)):
        axs[i].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Show the first labeled batch
show_batch(batches[0][0], batches[0][1])

In [ ]:
# Show each labeling batch (new labels only)
for name, data in batches[1:]:
    show_batch(name, data)

In [ ]:
# Summary table of all labels
from collections import Counter

print("Label distribution across all batches:\n")
for name, data in batches:
    if not data:
        continue
    counter = Counter(x for v in data.values() for x in v)
    print(f"--- {name} ({len(data)} images) ---")
    for label, count in sorted(counter.items()):
        print(f"  {label}: {count}")
    print()